# Flight Delay Analysis

In this lab, you will analyze a real-world dataset -- information about US flight delays in January 2016.

This dataset ships with two files. 
First, the `airline-format.html` file contains a brief description of the dataset, and the various data fields. For example, the `ArrDelay` field is the flight's arrival delay, in minutes. 
Second, the `airline-delays.csv` file is a comma-separated collection of flight records, one record per line.

## Parsing csv Data

You have to parse the CSV data. The header line provides the column names, and then each subsequent line can be parsed taking these into account. Python has a built-in `csv` module (unrelated to Spark) that you can use to parse CSV lines.

In [1]:
import csv
from io import StringIO

def parseLine(line, fieldnames):
    si = StringIO(line)
    read = csv.DictReader(si, fieldnames=fieldnames)
    return next(read)

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('FlightDelayAnalysisApp').getOrCreate()
sc = spark.sparkContext


In [3]:
rdd = sc.textFile("data/airline-delays.csv")
headerline = rdd.first()
print(headerline)

"Year","Quarter","Month","DayofMonth","DayOfWeek","FlightDate","UniqueCarrier","AirlineID","Carrier","TailNum","FlightNum","OriginAirportID","OriginAirportSeqID","OriginCityMarketID","Origin","OriginCityName","OriginState","OriginStateFips","OriginStateName","OriginWac","DestAirportID","DestAirportSeqID","DestCityMarketID","Dest","DestCityName","DestState","DestStateFips","DestStateName","DestWac","CRSDepTime","DepTime","DepDelay","DepDelayMinutes","DepDel15","DepartureDelayGroups","DepTimeBlk","TaxiOut","WheelsOff","WheelsOn","TaxiIn","CRSArrTime","ArrTime","ArrDelay","ArrDelayMinutes","ArrDel15","ArrivalDelayGroups","ArrTimeBlk","Cancelled","CancellationCode","Diverted","CRSElapsedTime","ActualElapsedTime","AirTime","Flights","Distance","DistanceGroup","CarrierDelay","WeatherDelay","NASDelay","SecurityDelay","LateAircraftDelay","FirstDepTime","TotalAddGTime","LongestAddGTime","DivAirportLandings","DivReachedDest","DivActualElapsedTime","DivArrDelay","DivDistance","Div1Airport","Div1A

In [12]:
fieldnames = filter(lambda field: len(list(field)) > 0,
                map(lambda field: field.strip('"'), headerline.split(',')))

fieldnames = list(fieldnames)
print(fieldnames)

['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'UniqueCarrier', 'AirlineID', 'Carrier', 'TailNum', 'FlightNum', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime', 'DivAirportLandings', 'DivReachedDest', 'Div

In [14]:
flights = rdd.filter(lambda line: line != headerline)  \
             .map(lambda line: parseLine(line, fieldnames))
flights.persist()
flights.take(1)

[OrderedDict([('Year', '2016'),
              ('Quarter', '1'),
              ('Month', '1'),
              ('DayofMonth', '6'),
              ('DayOfWeek', '3'),
              ('FlightDate', '2016-01-06'),
              ('UniqueCarrier', 'AA'),
              ('AirlineID', '19805'),
              ('Carrier', 'AA'),
              ('TailNum', 'N4YBAA'),
              ('FlightNum', '43'),
              ('OriginAirportID', '11298'),
              ('OriginAirportSeqID', '1129804'),
              ('OriginCityMarketID', '30194'),
              ('Origin', 'DFW'),
              ('OriginCityName', 'Dallas/Fort Worth, TX'),
              ('OriginState', 'TX'),
              ('OriginStateFips', '48'),
              ('OriginStateName', 'Texas'),
              ('OriginWac', '74'),
              ('DestAirportID', '11433'),
              ('DestAirportSeqID', '1143302'),
              ('DestCityMarketID', '31295'),
              ('Dest', 'DTW'),
              ('DestCityName', 'Detroit, MI'),
          

## Querying Flights and Delays

Now that you have the flight objects, it's time to perform a few queries and gather some useful information. Suppose you're in Boston, MA. Which airline has the most flights departing from Boston?


In [15]:
flightsByCarrier = flights.filter(
      lambda flight: flight['OriginCityName'] == "Boston, MA")    \
                           .map(lambda flight: flight['Carrier']) \
                           .countByValue()
sorted(flightsByCarrier.items(), key=lambda p: p[1], reverse=True)[0]

('B6', 3417)

Overall, which airline has the worst average delay? How bad was that delay?
**HINT**: Use `combineByKey`.

In [16]:
result = flights.filter(lambda f: f['ArrDelay'] != '') \
    .map(lambda f: (f['Carrier'], float(f['ArrDelay']))) \
    .combineByKey(lambda d: (d, 1),lambda s, d: (s[0]+d, s[1]+1),lambda s1, s2: (s1[0]+s2[0], s1[1]+s2[1])) \
    .map(lambda x: (x[0], x[1][0]/float(x[1][1]))) \
    .sortBy(lambda x: x[1])
    
#result.take(5)
for s, d in result.take(5):
    print("Carrier %s has average delay(in minutes) %.2f" % (s, d))
print("The worst one is: ", result.first())

Carrier AS has average delay(in minutes) -3.92
Carrier UA has average delay(in minutes) -2.89
Carrier F9 has average delay(in minutes) -1.86
Carrier HA has average delay(in minutes) -1.63
Carrier DL has average delay(in minutes) -0.84
The worst one is:  ('AS', -3.922802511415525)


## Living in Chicago, IL, what are the farthest 10 destinations that you could fly to?
(Note that our dataset contains only US domestic flights.)

In [17]:
flights.filter(lambda f: f['OriginCityName'] == "Chicago, IL")      \
       .map(lambda f: (f['DestCityName'], float(f['Distance'])))    \
       .distinct()                                                  \
       .sortBy(lambda x: x[1], ascending=False)                       \
       .take(10)

[('Honolulu, HI', 4243.0),
 ('Kahului, HI', 4184.0),
 ('Anchorage, AK', 2846.0),
 ('Charlotte Amalie, VI', 2116.0),
 ('San Juan, PR', 2072.0),
 ('San Francisco, CA', 1855.0),
 ('San Francisco, CA', 1846.0),
 ('Oakland, CA', 1844.0),
 ('San Jose, CA', 1838.0),
 ('Oakland, CA', 1836.0)]

## Suppose you're in New York, NY and are contemplating direct flights to San Francisco, CA. In terms of arrival delay, which airline has the best record on that route?

In [19]:
result = flights.filter(lambda flight: flight['OriginCityName'] == "New York, NY" and
                              flight['DestCityName'] == "San Francisco, CA" and
                              flight['ArrDelay'] != '')                       \
       .map(lambda flight: (flight['Carrier'], float(flight['ArrDelay'])))    \
       .reduceByKey(lambda a, b: a + b)                                       \
       .sortBy(lambda x: x[1]) 
       
print(result.first())
result.take(10)


('DL', -1702.0)


[('DL', -1702.0), ('B6', -168.0), ('AA', 1616.0), ('VX', 2097.0)]

## Suppose you live in San Jose, CA, and there don't seem to be many direct flights taking you to Boston, MA. Of all the 1-stop flights, which would be the best option in terms of average arrival delay? (It's OK to assume that every pair of flights from San Jose to X and from X to Boston is an option that you could use.)

 **NOTE**: To answer this question, you will probably need a cartesian product of the dataset with itself. Beside the fact that it's a fairly expensive operation, we haven't learned about multi-RDD operations yet. Still, you can explore the `join` RDD method, which applies to pair (key-value) RDDs.


In [20]:
flightsByDst = flights.filter(lambda f: f['OriginCityName'] == 'San Jose, CA')\
                      .map(lambda f: (f['DestCityName'], f))
flightsByOrg = flights.filter(lambda f: f['DestCityName'] == 'Boston, MA')    \
                      .map(lambda f: (f['OriginCityName'], f))

def addDelays(f1, f2):
    total = 0
    t1 = float(f1['ArrDelay']) if f1['ArrDelay'] != '' else 0
    t2 = float(f2['ArrDelay']) if f2['ArrDelay'] != '' else 0
    total = t1+t2
    return total

result = flightsByDst.join(flightsByOrg)                                       \
            .map(lambda x: (x[0], addDelays(x[1][0], x[1][1])))  \
            .combineByKey(lambda d: (d, 1),
                          lambda s, d: (s[0]+d, s[1]+1),
                          lambda s1, s2: (s1[0]+s2[0], s1[1]+s2[1]))  \
            .map(lambda x: (x[0], x[1][0]/float(x[1][1]))) \
            .sortBy(lambda x: x[1])
        
for s, d in result.take(5):
    print("1-stop city %s, the average delay(in minutes) %.2f" % (s, d))

1-stop city Minneapolis, MN, the average delay(in minutes) -8.52
1-stop city Houston, TX, the average delay(in minutes) -6.49
1-stop city Dallas/Fort Worth, TX, the average delay(in minutes) -5.13
1-stop city Portland, OR, the average delay(in minutes) -2.34
1-stop city Chicago, IL, the average delay(in minutes) 0.93


In [21]:
sc.stop()